# SqueezeNet - Turntable Classification

**Model:** SqueezeNet  
**Dataset:** Turntable (12 classes, 96x96)  
**Loss:** sparse_categorical_crossentropy  
**Epochs:** 40 | **Batch Size:** 32 | **Optimizer:** Adam (lr=0.001)

In [ ]:
import numpy as np
import h5py
import os
import time
import csv

os.environ['TF_ENABLE_ONEDNN_OPTS'] = '0'
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '2'

import tensorflow as tf
import keras
from keras.layers import (Conv2D, Concatenate, BatchNormalization, Activation,
                          Dense, GlobalAveragePooling2D, Input, MaxPooling2D)
from keras.models import Model
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score
from sklearn.utils.class_weight import compute_class_weight

tf.random.set_seed(42)
np.random.seed(42)

OUTPUT_DIR = '.'
OUTPUT_DIR = '.'

In [ ]:
# --- Load Data ---
print("Loading turntable data...")
DATA_DIR = '../data_down'
    x_train, y_train = f['x_train'][:], f['y_train'][:]
    x_test, y_test = f['x_test'][:], f['y_test'][:]
    class_names = [c.decode() if isinstance(c, bytes) else str(c) for c in f['class_names'][:]]

print(f"Train: {x_train.shape}, Test: {x_test.shape}, Classes: {len(class_names)}")

In [ ]:
# --- Model Definition ---
def fire_module(inp, s11, e11, e33):
    squeeze = Conv2D(s11, (1,1), padding='same', activation='relu')(inp)
    expand1x1 = Conv2D(e11, (1,1), padding='same', activation='relu')(squeeze)
    expand3x3 = Conv2D(e33, (3,3), padding='same', activation='relu')(squeeze)
    merged = Concatenate(axis=-1)([expand1x1, expand3x3])
    return BatchNormalization()(merged)

def squeezenet(input_shape, num_classes, base_squ_filt=8, base_exp_filt=32):
    inp = Input(shape=input_shape)
    x = Conv2D(8, (7,7), strides=(2,2), padding='same', activation='relu')(inp)
    x = BatchNormalization()(x)
    x = MaxPooling2D((2,2))(x)
    sq, ex = base_squ_filt, base_exp_filt
    x = fire_module(x, sq, ex, ex)
    x = fire_module(x, sq, ex, ex)
    x = fire_module(x, 2*sq, 2*ex, 2*ex)
    x = MaxPooling2D((2,2))(x)
    sq, ex = sq*2, ex*2
    x = fire_module(x, sq, ex, ex)
    x = fire_module(x, int(1.5*sq), int(1.5*ex), int(1.5*ex))
    x = fire_module(x, int(1.5*sq), int(1.5*ex), int(1.5*ex))
    x = fire_module(x, 2*sq, 2*ex, 2*ex)
    x = MaxPooling2D((2,2))(x)
    sq, ex = sq*2, ex*2
    x = fire_module(x, sq, ex, ex)
    x = Conv2D(num_classes, (5,5), padding='same', activation='relu')(x)
    x = BatchNormalization()(x)
    x = GlobalAveragePooling2D()(x)
    out = Activation('softmax')(x)
    return Model(inputs=inp, outputs=out, name='SqueezeNet')

model = squeezenet((96,96,1), len(class_names))
print(f"Parameters: {model.count_params():,}")

In [ ]:
# --- Class weights ---
weights = compute_class_weight('balanced', classes=np.unique(y_train), y=y_train)
class_weight = dict(enumerate(weights))

In [ ]:
# --- Train ---
model.compile(optimizer=keras.optimizers.Adam(1e-3),
              loss='sparse_categorical_crossentropy', metrics=['accuracy'])

print("\nTraining SqueezeNet on Turntable (40 epochs)...")
start = time.time()
history = model.fit(x_train, y_train, epochs=40, batch_size=32,
                    validation_data=(x_test, y_test), class_weight=class_weight, verbose=2)
elapsed = time.time() - start
print(f"Training time: {elapsed:.1f}s ({elapsed/60:.1f} min)")

In [ ]:
# --- Evaluate ---
y_pred = np.argmax(model.predict(x_test, verbose=0), axis=1)
acc = accuracy_score(y_test, y_pred)
print(f"\nTest Accuracy: {acc*100:.2f}%")
report = classification_report(y_test, y_pred, target_names=class_names, digits=3, zero_division=0)
print(report)

In [ ]:
# --- Save Model ---
OUTPUT_DIR = '.'

In [ ]:
# --- Save Single results.csv ---
report_dict = classification_report(y_test, y_pred, target_names=class_names,
                                     digits=4, zero_division=0, output_dict=True)
cm = confusion_matrix(y_test, y_pred)

OUTPUT_DIR = '.'
    w = csv.writer(f)

    # Section 1: Summary
    w.writerow(['## Summary'])
    w.writerow(['metric', 'value'])
    w.writerow(['model', 'SqueezeNet'])
    w.writerow(['dataset', 'turntable'])
    w.writerow(['num_classes', len(class_names)])
    w.writerow(['input_shape', '96x96x1'])
    w.writerow(['parameters', model.count_params()])
    w.writerow(['epochs', 40])
    w.writerow(['batch_size', 32])
    w.writerow(['loss_function', 'sparse_categorical_crossentropy'])
    w.writerow(['optimizer', 'Adam (lr=0.001)'])
    w.writerow(['class_weights', 'balanced'])
    w.writerow(['test_accuracy', round(float(acc), 4)])
    w.writerow(['paper_accuracy', 0.991])
    w.writerow(['training_time_sec', round(elapsed, 1)])
    w.writerow([])

    # Section 2: Training History
    w.writerow(['## Training History'])
    w.writerow(['epoch', 'loss', 'accuracy', 'val_loss', 'val_accuracy'])
    for i in range(len(history.history['loss'])):
        w.writerow([i+1,
                     round(history.history['loss'][i], 4),
                     round(history.history['accuracy'][i], 4),
                     round(history.history['val_loss'][i], 4),
                     round(history.history['val_accuracy'][i], 4)])
    w.writerow([])

    # Section 3: Per-Class Metrics
    w.writerow(['## Per-Class Metrics'])
    w.writerow(['class', 'precision', 'recall', 'f1_score', 'support'])
    for cls in class_names:
        m = report_dict[cls]
        w.writerow([cls, round(m['precision'],4), round(m['recall'],4),
                     round(m['f1-score'],4), int(m['support'])])
    for avg in ['macro avg', 'weighted avg']:
        m = report_dict[avg]
        w.writerow([avg, round(m['precision'],4), round(m['recall'],4),
                     round(m['f1-score'],4), int(m['support'])])
    w.writerow([])

    # Section 4: Confusion Matrix
    w.writerow(['## Confusion Matrix'])
    w.writerow([''] + class_names)
    for i, cls in enumerate(class_names):
        w.writerow([cls] + cm[i].tolist())

print(f"\nSaved: model.keras + results.csv to {OUTPUT_DIR}")